# 18 — Preference-Filtered SFT

After supervised fine-tuning (NB17), this notebook performs a second pass
over preference pairs (chosen vs. rejected). With `USE_PREFERENCE_LOSS = True`
(default, issue #413) it trains a real preference objective via
`preference_loss.py`: cross-entropy on the chosen response **plus a margin
hinge that pushes the chosen response's per-token log-probability above the
rejected one's**. The rejected samples are finally part of the loss, not just
the curation stage.

> **Fallback:** mlx-lm's `lora` CLI does not implement DPO. Set
> `USE_PREFERENCE_LOSS = False` to restore the previous chosen-only SFT
> (preference data used only to select what to train on).

## Data Sources

1. **NB15 rejects** — `data/04_validated/rejects_for_dpo.jsonl` — samples that
   looked like ARO but failed `aro check`.
2. **NB20 round rejects** — `data/rounds/round_N_rejects.jsonl` — samples that
   failed during iterative generation.
3. **NB10 repair rejects** — `data/03_raw_generated/repair_rejects.jsonl` —
   failed repair attempts during synthetic data generation.
4. **Fresh pairs** — N candidates per prompt with the SFT model, labelled
   chosen/rejected by `aro check`.

## Pair-building strategy

DPO-style triples `(prompt, chosen_response, rejected_response)` are still built
because we plan to use them once a real DPO trainer is available. Rejected
samples are filtered to keep only "close but wrong" — they must look like ARO
(contain `(` and `<`) but fail syntax validation. Pairs are kept only when the
chosen response is for the *same* instruction as the reject — random pairing
across unrelated prompts is rejected because it teaches the model wrong
answers for the prompt at hand.

**Input:**  SFT fused model from NB17 + reject files from NB15/NB20/NB10

**Output:** `../models/dpo/adapter/` — preference-SFT adapter
            `../models/dpo/fused/`   — merged model ready for packaging

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

In [ ]:
import json
import os
import re
import random
import subprocess
import sys
import tempfile
import importlib
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
import config; importlib.reload(config); from config import *

# ── Paths ──────────────────────────────────────────────────────────────────────
DPO_DATA_DIR   = DATA_ROOT / 'dpo'
DPO_DATA_DIR.mkdir(parents=True, exist_ok=True)

DPO_MODEL_DIR  = MODELS_DIR / 'dpo'
DPO_ADAPTER    = DPO_MODEL_DIR / 'adapter'
DPO_FUSED      = DPO_MODEL_DIR / 'fused'
DPO_ADAPTER.mkdir(parents=True, exist_ok=True)

# ── Find the best SFT model to start from ────────────────────────────────────
def _find_latest_fused():
    """Find the latest fused SFT model (from NB12 or NB12)."""
    for parent in [MODELS_DIR / 'iterative', FINETUNE_MODELS_DIR]:
        if not parent.exists():
            continue
        rounds = sorted(parent.glob('round_*/fused'), reverse=True)
        for r in rounds:
            if (r / 'config.json').exists():
                return str(r)
    # Fall back to base model
    return MODEL_ID

SFT_MODEL = _find_latest_fused()

# ── Feature flags ──────────────────────────────────────────────────────────
EMPTY_PENALTY_PAIRS = True    # issue #414 — synthetic (chosen, rejected="") pairs
SEMANTIC_GATE       = True    # issue #415 — judge chosen responses before training
USE_PREFERENCE_LOSS = True    # issue #413 — real preference loss via
                              # preference_loss.py; False = chosen-only SFT

print(f'SFT model:    {SFT_MODEL}')
print(f'Empty-content penalty pairs: {EMPTY_PENALTY_PAIRS}')
print(f'Semantic-alignment gate:     {SEMANTIC_GATE}')
print(f'Preference loss:             {USE_PREFERENCE_LOSS}')
print(f'DPO adapter:  {DPO_ADAPTER}')
print(f'DPO fused:    {DPO_FUSED}')

## 1. Collect rejected samples

In [ ]:
def load_rejects():
    """Load all rejected samples from NB12, NB12, NB10, and user session repairs."""
    rejects = []

    # NB12 (validation) rejects
    nb16_rejects = DATA_ROOT / '04_validated' / 'rejects_for_dpo.jsonl'
    if nb16_rejects.exists():
        with open(nb16_rejects) as f:
            for line in f:
                if line.strip():
                    rejects.append(json.loads(line))
        print(f'  NB12 rejects: {len(rejects)}')

    # NB12 (iterative loop) round rejects
    rounds_dir = DATA_ROOT / 'rounds'
    if rounds_dir.exists():
        n_round_rejects = 0
        for rfile in sorted(rounds_dir.glob('round_*_rejects.jsonl')):
            with open(rfile) as f:
                for line in f:
                    if line.strip():
                        rejects.append(json.loads(line))
                        n_round_rejects += 1
        if n_round_rejects:
            print(f'  NB12 round rejects: {n_round_rejects}')

    # NB10 repair rejects (failed attempts during generate_with_repair)
    repair_rejects = DATA_ROOT / '03_raw_generated' / 'repair_rejects.jsonl'
    if repair_rejects.exists():
        n_repair = 0
        with open(repair_rejects) as f:
            for line in f:
                if line.strip():
                    rejects.append(json.loads(line))
                    n_repair += 1
        if n_repair:
            print(f'  NB10 repair rejects: {n_repair}')

    # User session repair logs (.context.repairs.jsonl from `aro ask`).
    # Path: ARO-Train/.context.repairs.jsonl (the project root, not the script CWD).
    _user_repairs = ARO_ROOT / '.context.repairs.jsonl'
    if _user_repairs.exists():
        n_user = 0
        with open(_user_repairs) as f:
            for line in f:
                if line.strip():
                    entry = json.loads(line)
                    rejects.append({
                        'instruction': entry.get('error_prompt', ''),
                        'output': entry.get('broken_output', ''),
                        'task_type': 'debugging',
                        'validation_error': 'user_session_repair',
                    })
                    n_user += 1
        if n_user:
            print(f'  User session repairs: {n_user}')

    print(f'  Total rejects loaded: {len(rejects)}')
    return rejects


def is_close_but_wrong(output):
    """Filter: keep only rejects that look like ARO but have syntax errors.
    Drop empty, all-comments, or completely off-topic outputs."""
    if not output or len(output.strip()) < 30:
        return False
    # Must contain at least one feature set opening and one variable reference
    has_paren = '(' in output
    has_angle = '<' in output
    if not (has_paren and has_angle):
        return False
    # Reject if >70% comment lines
    lines = [l.strip() for l in output.splitlines() if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.startswith('(*'))
    if comment_lines / len(lines) > 0.7:
        return False
    return True


all_rejects = load_rejects()
filtered_rejects = [r for r in all_rejects if is_close_but_wrong(r.get('output', ''))]
print(f'After close-but-wrong filter: {len(filtered_rejects)} rejects')

# ── aro-check grounding for DPO pair construction ──────────────────────────
# A `is_close_but_wrong` reject can still be syntactically valid ARO (e.g.
# a comment-heavy file from NB12 round_rejects). Emitting that as
# "rejected" against a known-good "chosen" teaches the model that valid
# code is worse than other valid code — the opposite of what we want.
# Verify each pair through `_verify_pair_grounding` at construction time.

_ARO_FENCE_PAT = re.compile(r'```aro[ \t]*\r?\n?(.*?)\r?\n?```', re.DOTALL)


def _extract_aro_code(text):
    """Pull ARO source out of a markdown-fenced reply, or fall back to the
    raw text if no fence is present. Mirrors NB12 cell 10's extractor."""
    if not text:
        return ''
    blocks = _ARO_FENCE_PAT.findall(text)
    if blocks:
        return '\n'.join(b.strip() for b in blocks)
    return text.strip()


def aro_check_strict(text_or_code):
    """Return True iff the embedded ARO passes `aro check`.

    Returns False on failure, timeout, or empty input. Returns None ONLY
    if the `aro` binary isn't on PATH — callers should fail-closed in
    that case (don't ship unverified DPO pairs)."""
    code = _extract_aro_code(text_or_code)
    if not code.strip():
        return False
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(
                ['aro', 'check', tmp],
                capture_output=True, text=True, timeout=10,
            )
            return r.returncode == 0
    except FileNotFoundError:
        return None
    except subprocess.TimeoutExpired:
        return False


def _verify_pair_grounding(chosen_text, rejected_text):
    """A DPO pair is only useful if `chosen` passes `aro check` AND
    `rejected` fails it. Anything else is signal contamination:

      chosen fails, rejected passes  → trains the model that broken is better
      both pass                       → preference is arbitrary (style noise)
      both fail                       → preference is arbitrary (which-flavor-of-broken)

    Returns True iff the pair is properly grounded. Drops pairs where
    the `aro` binary is missing (fail-closed)."""
    chosen_ok   = aro_check_strict(chosen_text)
    rejected_ok = aro_check_strict(rejected_text)
    return chosen_ok is True and rejected_ok is False


## 2. Load valid samples (chosen responses)

In [ ]:
def load_valid_samples():
    """Load validated code_generation samples as 'chosen' responses.

    Reads from 04_validated/valid_samples.jsonl and 05_dataset/train.jsonl.
    These overlap because the validated set feeds into the training set, so
    we deduplicate by (instruction, output) hash to avoid biasing random
    sampling toward duplicated entries.
    """
    valid = []
    seen = set()  # (instr_first_200, output_first_200) — cheap content key

    def _add(instr, output, task_type):
        if not instr or not output:
            return
        key = (instr[:200], output[:200])
        if key in seen:
            return
        seen.add(key)
        valid.append({'instruction': instr, 'output': output, 'task_type': task_type})

    valid_file = DATA_ROOT / '04_validated' / 'valid_samples.jsonl'
    if valid_file.exists():
        with open(valid_file) as f:
            for line in f:
                if not line.strip():
                    continue
                s = json.loads(line)
                if s.get('task_type') in {'code_generation', 'translation', 'debugging'}:
                    _add(s.get('instruction', ''), s.get('output', ''), s.get('task_type', ''))

    train_file = DATA_ROOT / '05_dataset' / 'train.jsonl'
    if train_file.exists():
        with open(train_file) as f:
            for line in f:
                if not line.strip():
                    continue
                s = json.loads(line)
                if s.get('task_type') != 'code_generation':
                    continue
                msgs = s.get('messages', [])
                asst = [m['content'] for m in msgs if m['role'] == 'assistant']
                user = [m['content'] for m in msgs if m['role'] == 'user']
                if asst and user:
                    _add(user[0], asst[0], 'code_generation')

    print(f'Valid (chosen) samples: {len(valid)} (after dedup)')
    return valid

valid_samples = load_valid_samples()

## 3. Build DPO pairs

In [ ]:
def build_dpo_pairs(valid_samples, rejected_samples, add_empty_penalty=True):
    """Build (prompt, chosen, rejected) triples for preference SFT.

    Strategy: pair each reject only with a valid sample that has the **same
    instruction** (matched by 100-char prefix). Drop rejects with no matching
    valid sample — random pairing across unrelated prompts would teach the
    model wrong-answer-for-question patterns, which is positively harmful.

    Every candidate pair is then verified through `_verify_pair_grounding`:
    the `chosen` reply must pass `aro check` and the `rejected` reply must
    fail it. Pairs where both pass (style noise), both fail (which-flavor-
    of-broken is irrelevant), or the labels are inverted (broken-is-better)
    are dropped.

    Round-3 addition: after the grounded pairs are built, append synthetic
    "empty-content penalty" pairs where the rejected response is the
    literal empty string. The round-2 model collapsed to empty content on
    79% of free-form prompts; this teaches the model that empty is always
    worse than valid code, regardless of prompt phrasing.
    """
    pairs = []

    # Index valid by instruction prefix for matching
    valid_by_prefix = {}
    for v in valid_samples:
        instr = v.get('instruction', '')[:100]
        if instr:
            valid_by_prefix[instr] = v

    # Build system prompt
    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)

    matched     = 0
    no_match    = 0
    identical   = 0
    ungrounded  = 0   # chosen doesn't pass or rejected doesn't fail aro check
    aro_missing = False

    for rej in rejected_samples:
        rej_instr  = rej.get('instruction', '')
        rej_output = rej.get('output', '')
        if not rej_instr or not rej_output:
            continue

        prefix = rej_instr[:100]
        chosen_sample = valid_by_prefix.get(prefix)

        if chosen_sample is None:
            no_match += 1
            continue

        chosen_output = chosen_sample['output']
        if chosen_output.strip() == rej_output.strip():
            identical += 1
            continue

        if not _verify_pair_grounding(chosen_output, rej_output):
            ungrounded += 1
            if aro_check_strict('(stub: x) { Return an <OK: status> for <x>. }') is None:
                aro_missing = True
            continue

        pairs.append({
            'prompt': [
                {'role': 'system', 'content': sys_prompt},
                {'role': 'user', 'content': rej_instr},
            ],
            'chosen':   [{'role': 'assistant', 'content': chosen_output}],
            'rejected': [{'role': 'assistant', 'content': rej_output}],
        })
        matched += 1

    # Empty-content penalty (issue #414 — default-on via EMPTY_PENALTY_PAIRS):
    # synthesise (chosen=valid, rejected="") pairs from a sample of valid
    # examples. Guards against the round-2 collapse where a large share of
    # free-form prompts returned empty output. Capped at max(50, len(natural
    # pairs)) so the synthetic signal doesn't drown out the grounded ones.
    empty_added = 0
    if add_empty_penalty:
        import random as _rnd
        _rnd.seed(0)
        _empty_targets = list(valid_by_prefix.values())
        _rnd.shuffle(_empty_targets)
        _cap = min(len(_empty_targets), max(50, len(pairs)))
        for _v in _empty_targets[:_cap]:
            _instr = _v.get('instruction', '')
            _chosen = _v.get('output', '')
            if not _instr or not _chosen:
                continue
            pairs.append({
                'prompt': [
                    {'role': 'system', 'content': sys_prompt},
                    {'role': 'user', 'content': _instr},
                ],
                'chosen':   [{'role': 'assistant', 'content': _chosen}],
                'rejected': [{'role': 'assistant', 'content': ''}],
            })
            empty_added += 1

    print(f'DPO pairs built: {len(pairs)}')
    print(f'  Matched + grounded by `aro check`: {matched}')
    print(f'  Dropped (no matching valid):       {no_match}')
    print(f'  Dropped (chosen == rejected):      {identical}')
    print(f'  Dropped (ungrounded — chosen failed or rejected passed): {ungrounded}')
    print(f'  Empty-content penalty pairs added:  {empty_added}')
    if aro_missing:
        print('  WARNING: `aro` binary not on PATH — every pair was dropped fail-closed.')
        print('           Install aro or build the release binary before running NB12.')
    return pairs

dpo_pairs = build_dpo_pairs(valid_samples, filtered_rejects,
                            add_empty_penalty=EMPTY_PENALTY_PAIRS)


## 4. Generate fresh DPO pairs from SFT model

In [ ]:
def aro_check(code):
    """Run aro check on code. Returns True if valid, False otherwise."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp)
            (p / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', str(p)],
                               capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return False


# Tolerate optional language tag whitespace, optional newline after the tag,
# and a leading newline before the closing fence. Falls back to whole text.
_ARO_FENCE_RE = re.compile(r'```aro[ \t]*\r?\n?(.*?)\r?\n?```', re.DOTALL)

def extract_aro_code(text):
    """Extract ARO code from markdown-fenced or raw text."""
    blocks = _ARO_FENCE_RE.findall(text)
    if blocks:
        return '\n'.join(b.strip() for b in blocks)
    return text.strip()


def generate_fresh_dpo_pairs(model, tokenizer, generate_fn, sampler_fn, n_prompts=50, n_candidates=4):
    """Generate N candidates per prompt, use aro check to label chosen/rejected.

    Pair candidates **1-to-1** (zip the shorter list) so a prompt with 4 chosen
    + 4 rejected contributes 4 pairs, not 16. The previous Cartesian product
    overweighted prompts where the model happened to produce diverse outputs.
    """
    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)
    fresh_pairs = []

    # Use a variety of prompts
    prompts = [
        'Write an ARO feature set that retrieves a user by ID and returns an OK response.',
        'Write an ARO Application-Start that starts an HTTP server and keeps it alive.',
        'Write an ARO event handler that processes OrderCreated events and logs the order.',
        'Write an ARO feature set that reads a JSON file and iterates over its items.',
        'Write an ARO API endpoint that creates a new item in a repository.',
        'Write an ARO feature set that fetches data from an external URL and transforms it.',
        'Write an ARO feature set that filters a list of items by price > 100.',
        'Write an ARO Application-Start that watches a directory for file changes.',
        'Write an ARO feature set that computes the hash of a password and stores it.',
        'Write an ARO feature set that splits a CSV line and logs each field.',
        'Write an ARO WebSocket handler that broadcasts messages to all connections.',
        'Write an ARO feature set that uses a state machine to process orders.',
        'Write an ARO feature set that merges two collections and returns the union.',
        'Write an ARO feature set that renders an HTML template with user data.',
        'Write an ARO feature set that schedules a task to run every 5 minutes.',
        'Write an ARO feature set that extracts query parameters from a request.',
        'Write an ARO feature set that uses date arithmetic to compute a deadline.',
        'Write an ARO feature set that validates an email format and returns errors.',
        'Write an ARO feature set that copies a file from source to destination.',
        'Write an ARO feature set that emits a custom event with metadata.',
    ]

    # Extend with random variations
    domains = ['blog', 'inventory', 'payment', 'notification', 'report',
               'search', 'analytics', 'auth', 'chat', 'monitoring']
    for d in domains:
        prompts.append(f'Write a complete ARO HTTP API for a {d} service with CRUD operations.')

    random.shuffle(prompts)
    prompts = prompts[:n_prompts]

    # Create a sampler with higher temperature for diversity
    sampler = sampler_fn(temp=0.7)

    for i, prompt_text in enumerate(prompts):
        messages = [
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user', 'content': prompt_text},
        ]
        chat_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        chosen_list, rejected_list = [], []

        for _ in range(n_candidates):
            try:
                response = generate_fn(
                    model, tokenizer, prompt=chat_prompt,
                    max_tokens=1024, verbose=False,
                    sampler=sampler,
                )
                code = extract_aro_code(response)
                if aro_check(code):
                    chosen_list.append(response)
                elif is_close_but_wrong(response):
                    rejected_list.append(response)
            except Exception as e:
                print(f'  Generation error: {e}')
                continue

        # 1-to-1 pairing — pair the i-th chosen with the i-th rejected.
        for chosen, rejected in zip(chosen_list, rejected_list):
            fresh_pairs.append({
                'prompt': messages,
                'chosen': [{'role': 'assistant', 'content': chosen}],
                'rejected': [{'role': 'assistant', 'content': rejected}],
            })

        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(prompts)} prompts processed, {len(fresh_pairs)} pairs so far')

    print(f'Fresh DPO pairs generated: {len(fresh_pairs)}')
    return fresh_pairs


# Load model and generate
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()
print(f'Loading SFT model from {SFT_MODEL}...')
model, tokenizer = load_fn(SFT_MODEL)
from config import patch_qwen3_chat_template
patch_qwen3_chat_template(tokenizer)

print('Model loaded.')

fresh_pairs = generate_fresh_dpo_pairs(
    model, tokenizer, generate_fn, make_sampler_fn,
    n_prompts=30,     # adjust based on time budget
    n_candidates=4,
)

## 4b. Semantic-Alignment Gate on Chosen Responses (issue #415)

`aro check` grounding validates pairs only *technically*: chosen passes,
rejected fails. Nothing so far checks that the chosen response actually
solves its instruction — a valid-but-wrong implementation would be enshrined
as the "preferred" answer, and preference training amplifies whatever the
chosen side encodes.

This cell runs `config.semantic_alignment_check` (the same conservative judge
NB10 uses) over the **chosen** side of every pair, using the already-loaded
SFT model as judge. Misaligned pairs are dropped and the drop count logged.
Identical (instruction, chosen) combinations are judged once.

In [ ]:
if not SEMANTIC_GATE:
    print('Semantic-alignment gate disabled — keeping all pairs.')
else:
    _judge_sampler = make_sampler_fn(temp=0.0)

    def _judge_chat(messages, max_tokens=160, temp=None):
        _text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        return generate_fn(model, tokenizer, prompt=_text,
                           max_tokens=max_tokens, sampler=_judge_sampler,
                           verbose=False)

    _judge_memo = {}

    def _gate_pairs(pairs, label):
        kept, dropped = [], 0
        for _i, _pair in enumerate(pairs):
            _instr  = _pair['prompt'][-1]['content']
            _chosen = _pair['chosen'][0]['content']
            _key = (_instr[:200], _chosen[:200])
            if _key not in _judge_memo:
                _judge_memo[_key] = semantic_alignment_check(
                    _instr, _chosen, _judge_chat)
            _aligned, _reason = _judge_memo[_key]
            if _aligned:
                kept.append(_pair)
            else:
                dropped += 1
                if dropped <= 5:
                    print(f'  [{label}] gate drop: {_reason[:110]}')
            if (_i + 1) % 25 == 0:
                print(f'  [{label}] judged {_i + 1}/{len(pairs)} '
                      f'(kept {len(kept)}, dropped {dropped})', flush=True)
        return kept, dropped

    dpo_pairs, _d1 = _gate_pairs(dpo_pairs, 'reject-matched')
    fresh_pairs, _d2 = _gate_pairs(fresh_pairs, 'fresh')
    print(f'Semantic gate: dropped {_d1} reject-matched + {_d2} fresh pairs '
          f'({_d1 + _d2} total; {len(dpo_pairs) + len(fresh_pairs)} kept)')

## 5. Save DPO dataset

In [ ]:
all_dpo_pairs = dpo_pairs + fresh_pairs
random.shuffle(all_dpo_pairs)

# mlx-lm 0.31.1 doesn't have native DPO support.
# Convert preference pairs to chat format: train on chosen responses only
# (preference-filtered SFT).  The rejected samples still help at the
# collection stage — they ensure we only train on outputs that beat a
# known-bad alternative.

def dpo_pair_to_chat(pair):
    """Convert a DPO (prompt, chosen, rejected) triple to chat messages format."""
    messages = []
    # prompt is a list of message dicts
    for m in pair['prompt']:
        messages.append(m)
    # chosen is a list with one assistant message
    for m in pair['chosen']:
        messages.append(m)
    return {'messages': messages}

chat_samples = [dpo_pair_to_chat(p) for p in all_dpo_pairs]

# Complete-program filter on the CHOSEN responses. A preference pair where
# the "winner" is itself a fragment teaches nothing structural; drop them.
# This is a heavier filter than NB12 because preference SFT amplifies
# whatever pattern the chosen response carries.
_pre_n = len(chat_samples)
chat_samples = [
    s for s in chat_samples
    if is_complete_program(s['messages'][-1].get('content', ''))
]
print(f'Complete-program filter on chosen responses: '
      f'kept {len(chat_samples)}/{_pre_n} '
      f'(dropped {_pre_n - len(chat_samples)} fragment-only winners)')

# Split 90/10 train/valid
split_idx = int(len(chat_samples) * 0.9)
train_samples = chat_samples[:split_idx]
valid_samples_dpo = chat_samples[split_idx:]

# Save in mlx-lm chat format
for name, samples in [('train', train_samples), ('valid', valid_samples_dpo)]:
    out_path = DPO_DATA_DIR / f'{name}.jsonl'
    with open(out_path, 'w') as f:
        for sample in samples:
            f.write(json.dumps(sample) + '\n')
    print(f'Saved {len(samples)} samples to {out_path}')

# Also save the raw DPO pairs for reference
raw_path = DPO_DATA_DIR / 'dpo_pairs_raw.jsonl'
with open(raw_path, 'w') as f:
    for pair in all_dpo_pairs:
        f.write(json.dumps(pair) + '\n')
print(f'Saved {len(all_dpo_pairs)} raw DPO pairs to {raw_path}')

# ── Preference-format split for preference_loss.py (issue #413) ──────────────
# Same shuffled order and 90/10 split as the chat files, but keeping BOTH
# sides of each pair. preference_loss.py reads train.jsonl/valid.jsonl from
# this directory (and itself skips pairs that would truncate).
PREF_DATA_DIR = DPO_DATA_DIR / 'pref'
PREF_DATA_DIR.mkdir(parents=True, exist_ok=True)
_pref_split = int(len(all_dpo_pairs) * 0.9)
for _name, _pref_samples in [('train', all_dpo_pairs[:_pref_split]),
                             ('valid', all_dpo_pairs[_pref_split:])]:
    _pp = PREF_DATA_DIR / f'{_name}.jsonl'
    with open(_pp, 'w') as f:
        for _pair in _pref_samples:
            f.write(json.dumps(_pair) + '\n')
    print(f'Saved {len(_pref_samples)} preference pairs to {_pp}')

print(f'\nTotal samples: {len(chat_samples)} (train={len(train_samples)}, valid={len(valid_samples_dpo)})')

In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)

with open(_run_dir / '18_preference_sft.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['pair_type', 'count', 'split'])
    w.writerow(['raw_rejects', len(all_rejects), ''])
    w.writerow(['close_but_wrong', len(filtered_rejects), ''])
    w.writerow(['matched_pairs', len(dpo_pairs), ''])
    w.writerow(['fresh_pairs', len(fresh_pairs), ''])
    w.writerow(['total_dpo', len(all_dpo_pairs), ''])
    w.writerow(['train', len(train_samples), 'train'])
    w.writerow(['valid', len(valid_samples_dpo), 'valid'])

print(f'Saved: {_run_dir / "18_preference_sft.csv"}')

## Pre-flight: drop oversized samples

mlx-lm silently truncates anything longer than `MAX_SEQ_LEN`, which corrupts
training (assistant responses get cut mid-sentence). Truncation on chosen
responses is a likely cause of the NaN-loss blowup observed on the 2026-04-30
NB12 run, so we tokenize each sample with the SFT model's tokenizer and drop
oversized ones into a filtered copy that the training cell uses.

In [ ]:
import warnings as _warnings

MAX_SEQ_LEN = 4096    # must match the --max-seq-length passed to mlx_lm.lora

# Reuse the SFT tokenizer already loaded above (`tokenizer`).
DPO_FILTERED_DIR = DPO_DATA_DIR.parent / 'dpo_filtered'
DPO_FILTERED_DIR.mkdir(parents=True, exist_ok=True)

def _sample_token_count(rec):
    """Token length for an mlx-lm chat-format sample."""
    msgs = rec.get('messages')
    if msgs:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    else:
        text = rec.get('prompt', '') + rec.get('completion', '') or rec.get('text', '')
    return len(tokenizer.encode(text, add_special_tokens=False))

def _filter_file(name):
    src = DPO_DATA_DIR / name
    dst = DPO_FILTERED_DIR / name
    if not src.exists():
        print(f'  {name}: missing — skipping')
        return [], []
    kept_lines, dropped = [], []
    with open(src) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            n = _sample_token_count(rec)
            if n <= MAX_SEQ_LEN:
                kept_lines.append(line)
            else:
                dropped.append((n, rec))
    with open(dst, 'w') as f:
        f.write('\n'.join(kept_lines) + ('\n' if kept_lines else ''))
    return kept_lines, dropped

print(f'Filtering samples > {MAX_SEQ_LEN} tokens...')
_train_kept, _train_dropped = _filter_file('train.jsonl')
_valid_kept, _valid_dropped = _filter_file('valid.jsonl')

_total_in   = (len(_train_kept) + len(_train_dropped)) + (len(_valid_kept) + len(_valid_dropped))
_total_drop = len(_train_dropped) + len(_valid_dropped)
_drop_pct   = (_total_drop / _total_in * 100) if _total_in else 0

print(f'\ntrain.jsonl: kept {len(_train_kept):>4}, dropped {len(_train_dropped):>3}')
print(f'valid.jsonl: kept {len(_valid_kept):>4}, dropped {len(_valid_dropped):>3}')
print(f'overall:     dropped {_total_drop} / {_total_in}  ({_drop_pct:.1f}%)')

if _total_drop:
    _all_dropped = _train_dropped + _valid_dropped
    _longest = max(n for n, _ in _all_dropped)
    print(f'\nLongest dropped: {_longest} tokens  ({_longest / MAX_SEQ_LEN:.2f}x the limit)')

print(f'\nTraining will read from: {DPO_FILTERED_DIR}')

## 6. Run Preference Training

With `USE_PREFERENCE_LOSS = True` (default), training runs through
`preference_loss.py` — a small custom MLX loop (issue #413):

    loss = CE(chosen) + beta * max(0, margin − (logp(chosen) − logp(rejected)))

so the model learns both "produce valid ARO" (the CE anchor) and "prefer this
solution over that weaker one" (the hinge). Empty-rejected pairs (issue #414)
directly penalise confidence in immediate-EOS — the round-2 collapse mode.
With the flag off, the legacy chosen-only SFT via `mlx_lm lora` runs instead.

Regularisation now matches NB17 (issue #416): AdamW weight decay 0.01 and
LoRA rank 8 (previously rank 16 was *displayed* but never passed to the
trainer), plus cosine LR decay (issue #412). Both paths print the same
`Iter N: Train/Val loss` lines, so the guards below work unchanged:

- **NaN guard** — abort immediately if `Train loss nan` is reported.
- **Best-checkpoint tracking** — the checkpoint closest to the lowest
  val_loss is promoted to `adapters.safetensors` after training.

In [ ]:
import math

BATCH_SIZE    = 1
GRAD_ACCUM    = 8
LORA_LAYERS   = 8       # fewer than NB17's 16: second pass on an already
                        # fine-tuned model over a much smaller curated set —
                        # less capacity needed, smaller memorisation surface
LORA_RANK     = 8       # aligned with NB17 (issue #416) — was 16 and, worse,
                        # never actually passed to the trainer (mlx default 8)
LEARNING_RATE = 5e-6    # lower than SFT — preference data is curated
WEIGHT_DECAY  = 0.01    # same AdamW weight decay as NB17 (issue #416):
                        # the smaller dataset is MORE memorisation-prone
DPO_ITERS     = 200
SAVE_EVERY    = 50

# Preference-loss knobs (issue #413) — see preference_loss.py
PREF_MARGIN   = 0.5     # required log-prob/token gap chosen vs rejected
PREF_BETA     = 0.3     # weight of the ranking hinge term

# Stability guards (mirrors NB12)
LOSS_EXPLODE_THRESHOLD = 8.0

train_iters, train_losses = [], []
val_iters,   val_losses   = [], []
_loss_exploded     = False
_loss_nan          = False
_training_failed   = False  # high-level signal for downstream cells (fuse, smoke)
_first_train_loss  = None   # first non-NaN train loss observed (sanity check)
_first_nan_iter    = None   # iter where NaN first appeared (for the chart)
_first_explode_iter = None  # iter where loss > LOSS_EXPLODE_THRESHOLD

if len(all_dpo_pairs) < 10:
    print('⚠  Too few DPO pairs (<10). Skipping training.')
    print('   Run the full pipeline first to accumulate rejects, then re-run this notebook.')
    _training_failed = True
else:
    # Shared regularisation config (issue #416): the fallback CLI path reads
    # it via --config; the preference path receives the same values as flags.
    import yaml
    from train_utils import lr_schedule_config
    _dpo_cfg = {
        'optimizer': 'adamw',
        'optimizer_config': {'adamw': {'weight_decay': float(WEIGHT_DECAY)}},
        'lora_parameters': {'rank': LORA_RANK, 'dropout': 0.0, 'scale': 20.0},
        # Cosine LR decay (issue #412)
        'lr_schedule': lr_schedule_config(LEARNING_RATE, DPO_ITERS,
                                          warmup=min(20, DPO_ITERS // 10)),
    }
    _dpo_cfg_path = DPO_ADAPTER / 'dpo_config.yaml'
    with open(_dpo_cfg_path, 'w') as f:
        yaml.safe_dump(_dpo_cfg, f)

    if USE_PREFERENCE_LOSS:
        # Real preference objective (issue #413): CE on chosen + margin hinge
        # against rejected — the rejected samples finally earn their
        # collection cost. Output-line format matches mlx_lm.lora, so the
        # NaN/explosion monitoring below works unchanged.
        train_cmd = [
            sys.executable, str(SCRIPT_DIR / 'preference_loss.py'),
            '--model',                   SFT_MODEL,
            '--data',                    str(PREF_DATA_DIR),
            '--adapter-path',            str(DPO_ADAPTER),
            '--iters',                   str(DPO_ITERS),
            '--batch-size',              str(BATCH_SIZE),
            '--grad-accumulation-steps', str(GRAD_ACCUM),
            '--num-layers',              str(LORA_LAYERS),
            '--lora-rank',               str(LORA_RANK),
            '--learning-rate',           str(LEARNING_RATE),
            '--weight-decay',            str(WEIGHT_DECAY),
            '--lr-schedule',             'cosine',
            '--margin',                  str(PREF_MARGIN),
            '--beta',                    str(PREF_BETA),
            '--max-seq-length',          str(MAX_SEQ_LEN),
            '--save-every',              str(SAVE_EVERY),
            '--steps-per-eval',          '25',
        ]
        print('Starting preference-loss training (chosen + rejected)...')
    else:
        train_cmd = [
            sys.executable, '-m', 'mlx_lm', 'lora',
            '--config',                  str(_dpo_cfg_path),
            '--model',                   SFT_MODEL,
            '--data',                    str(DPO_FILTERED_DIR),
            '--train',
            '--batch-size',              str(BATCH_SIZE),
            '--grad-accumulation-steps', str(GRAD_ACCUM),
            '--num-layers',              str(LORA_LAYERS),
            '--learning-rate',           str(LEARNING_RATE),
            '--iters',                   str(DPO_ITERS),
            '--save-every',              str(SAVE_EVERY),
            '--val-batches',             '10',
            '--adapter-path',            str(DPO_ADAPTER),
            '--max-seq-length',          str(MAX_SEQ_LEN),
            '--mask-prompt',
        ]
        print('Starting preference-filtered SFT (chosen-only fallback)...')
    print(' '.join(train_cmd))
    print()

    _train_re = re.compile(
        r'Iter\s+(\d+):\s+Train loss\s+([-\w.]+)'
    )
    _val_re   = re.compile(r'Iter\s+(\d+):\s+Val loss\s+([-\w.]+)')

    def _is_nan(s):
        try:
            return math.isnan(float(s))
        except (TypeError, ValueError):
            return s.strip().lower() == 'nan'

    proc = subprocess.Popen(
        train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for raw_line in proc.stdout:
            line = raw_line.rstrip()
            print(line)

            m_train = _train_re.search(line)
            m_val   = _val_re.search(line)

            if m_train:
                it = int(m_train.group(1))
                loss_str = m_train.group(2)
                if _is_nan(loss_str):
                    _loss_nan = True
                    if _first_nan_iter is None:
                        _first_nan_iter = it
                    print(f'\n🚨  NaN train loss at iter {it} — aborting. '
                          f'Likely cause: truncated samples or LR too high.')
                    proc.terminate()
                    break
                loss = float(loss_str)
                if _first_train_loss is None:
                    _first_train_loss = loss
                train_iters.append(it)
                train_losses.append(loss)

                if loss > LOSS_EXPLODE_THRESHOLD:
                    _loss_exploded = True
                    if _first_explode_iter is None:
                        _first_explode_iter = it
                    print(f'\n🚨  LOSS EXPLOSION at iter {it}: train_loss={loss:.3f} '
                          f'> {LOSS_EXPLODE_THRESHOLD}. Aborting.')
                    proc.terminate()
                    break

            elif m_val:
                it = int(m_val.group(1))
                loss_str = m_val.group(2)
                if _is_nan(loss_str):
                    _loss_nan = True
                    print(f'\n🚨  NaN val loss at iter {it} — aborting.')
                    proc.terminate()
                    break
                val_iters.append(it)
                val_losses.append(float(loss_str))

    finally:
        proc.wait()

    if _loss_nan:
        _training_failed = True
        print(f'\n⚠  Training aborted due to NaN loss. The saved adapter is '
              f'unusable; skip the fuse step and re-run with a lower LR or '
              f'tighter MAX_SEQ_LEN.')
    elif _loss_exploded:
        _training_failed = True
        print(f'\n⚠  Training aborted: loss explosion. Reduce LEARNING_RATE '
              f'(currently {LEARNING_RATE:.0e}).')
    elif proc.returncode not in (0, -15):  # -15 = SIGTERM (from terminate())
        _training_failed = True
        print(f'\n⚠  Training exited with code {proc.returncode}')
    else:
        print(f'\n✓  Preference training complete. Adapter saved to: {DPO_ADAPTER}')
        if train_losses:
            print(f'  Final train_loss: {train_losses[-1]:.4f}')
        if val_losses:
            best_v = min(val_losses)
            best_vi = val_iters[val_losses.index(best_v)]
            print(f'  Best   val_loss:  {best_v:.4f}  (at iter {best_vi})')

## 6b. Select best checkpoint

Pick the saved checkpoint closest to the lowest val_loss and promote it to
`adapters.safetensors` (the file the fuse step reads). Skipped when training
failed — we don't want to fuse a NaN adapter.

In [ ]:
import glob, shutil

if _training_failed:
    print('Skipping checkpoint selection: training failed.')
else:
    _ckpt_files = sorted(glob.glob(str(DPO_ADAPTER / '*_adapters.safetensors')))
    _final_adapter = DPO_ADAPTER / 'adapters.safetensors'

    print(f'Adapter directory: {DPO_ADAPTER}')
    print(f'Checkpoints found: {len(_ckpt_files)}')

    _best_ckpt = None
    _best_reason = 'no checkpoints available'

    if val_losses and _ckpt_files:
        _best_val = min(val_losses)
        _best_val_iter = val_iters[val_losses.index(_best_val)]

        _ckpt_iters = []
        for cf in _ckpt_files:
            m = re.match(r'^(\d+)_adapters\.safetensors$', Path(cf).name)
            if m:
                _ckpt_iters.append((int(m.group(1)), cf))

        if _ckpt_iters:
            _ckpt_iters.sort(key=lambda x: x[0])
            _closest = min(_ckpt_iters, key=lambda x: abs(x[0] - _best_val_iter))
            _best_ckpt_iter, _best_ckpt = _closest
            _best_reason = (f'best val_loss={_best_val:.4f} at iter {_best_val_iter}, '
                            f'closest checkpoint at iter {_best_ckpt_iter}')

            # If the closest checkpoint == final iteration, the existing
            # adapters.safetensors is already correct.
            if _final_adapter.exists() and _best_ckpt_iter == _ckpt_iters[-1][0] \
                    and abs(DPO_ITERS - _best_val_iter) <= abs(_best_ckpt_iter - _best_val_iter):
                _best_ckpt = None
                _best_reason = (f'final adapter is already the best '
                                f'(val_loss={_best_val:.4f} at iter {_best_val_iter})')

    elif not val_losses and _ckpt_files:
        _best_ckpt = _ckpt_files[-1]
        _best_reason = 'no val_loss data; using last checkpoint'

    if _best_ckpt is not None:
        print(f'\nBest checkpoint: {Path(_best_ckpt).name}')
        print(f'  Reason: {_best_reason}')
        shutil.copy2(_best_ckpt, _final_adapter)
        print(f'  Promoted to adapters.safetensors')
    else:
        print(f'\nNo checkpoint copy needed: {_best_reason}')

## 7. Fuse DPO adapter

In [ ]:
if _training_failed:
    print('⚠  Skipping fuse: training failed (see above). The adapter is unusable.')
elif (DPO_ADAPTER / 'adapters.safetensors').exists():
    fuse_cmd = [
        sys.executable, '-m', 'mlx_lm', 'fuse',
        '--model',        local_model_path(SFT_MODEL),
        '--adapter-path', str(DPO_ADAPTER),
        '--save-path',    str(DPO_FUSED),
    ]

    print('Fusing preference-SFT adapter...')
    print(' '.join(fuse_cmd))
    result = subprocess.run(fuse_cmd)

    if result.returncode != 0:
        raise RuntimeError(f'Fuse failed with exit code {result.returncode}')

    print(f'✓  Fused model saved to: {DPO_FUSED}')
else:
    print('⚠  No adapter found — skipping fuse.')

## 7b. Round Metadata + Experiment Record (issues #416, #422)

`meta.json` makes the regularisation choices auditable and lets NB20's
`_meta_says_failed()` automatically skip a failed preference model. The run
is also recorded in the local `Train/experiments.db` (and mirrored to W&B
when `WANDB_PROJECT` is set).

In [ ]:
_dpo_meta = {
    'base_model':          SFT_MODEL,
    'loss':                ('preference (CE + margin hinge)'
                            if USE_PREFERENCE_LOSS else 'chosen-only SFT'),
    'use_preference_loss': bool(USE_PREFERENCE_LOSS),
    'margin':              PREF_MARGIN,
    'beta':                PREF_BETA,
    'iters':               DPO_ITERS,
    'batch_size':          BATCH_SIZE,
    'grad_accum':          GRAD_ACCUM,
    'lora_layers':         LORA_LAYERS,
    'lora_rank':           LORA_RANK,
    'learning_rate':       LEARNING_RATE,
    'weight_decay':        WEIGHT_DECAY,
    'lr_schedule':         'cosine',
    'max_seq_len':         MAX_SEQ_LEN,
    'n_pairs':             len(all_dpo_pairs),
    'empty_penalty_pairs': bool(EMPTY_PENALTY_PAIRS),
    'semantic_gate':       bool(SEMANTIC_GATE),
    'training_failed':     bool(_training_failed),
}
DPO_MODEL_DIR.mkdir(parents=True, exist_ok=True)
with open(DPO_MODEL_DIR / 'meta.json', 'w') as f:
    json.dump(_dpo_meta, f, indent=2)
print(f'Saved: {DPO_MODEL_DIR / "meta.json"}')

# ── Structured experiment tracking (issue #422) ──────────────────────────────
from experiment_db import record_run, DEFAULT_DB_PATH
_run_id = record_run(
    'NB18', config=_dpo_meta,
    metrics={
        'best_val_loss':    min(val_losses) if val_losses else None,
        'final_train_loss': train_losses[-1] if train_losses else None,
        'training_failed':  bool(_training_failed),
        'n_pairs':          len(all_dpo_pairs),
    },
    artifacts={'adapter': str(DPO_ADAPTER), 'fused': str(DPO_FUSED),
               'meta': str(DPO_MODEL_DIR / 'meta.json')})
print(f'Experiment recorded: run #{_run_id} in {DEFAULT_DB_PATH}')

## 8. Smoke test

In [ ]:
if _training_failed:
    print('⚠  Skipping smoke test: training failed.')
elif DPO_FUSED.exists() and (DPO_FUSED / 'config.json').exists():
    print(f'Loading preference-SFT model from {DPO_FUSED}...')
    dpo_model, dpo_tokenizer = load_fn(str(DPO_FUSED))

    from config import patch_qwen3_chat_template
    patch_qwen3_chat_template(dpo_tokenizer)

    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)

    test_messages = [
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Write an ARO feature set that retrieves a user by ID and returns an OK response.'},
    ]
    prompt = dpo_tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    response = generate_fn(dpo_model, dpo_tokenizer, prompt=prompt, max_tokens=300, verbose=False)

    print('\n=== Smoke Test ===')
    print(response)

    code = extract_aro_code(response)
    if aro_check(code):
        print('\n✓  Syntax check passed')
    else:
        print('\n⚠  Syntax check failed')
else:
    print('No fused model available for smoke test.')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '18_preference_sft.png'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: pair composition ───────────────────────────────────────────
_labels = ['Raw\nrejects', 'Close-but-\nwrong', 'Matched\npairs', 'Fresh\npairs', 'Total']
_values = [len(all_rejects), len(filtered_rejects), len(dpo_pairs), len(fresh_pairs), len(all_dpo_pairs)]
_colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#2c3e50']
bars = ax1.bar(_labels, _values, color=_colors, edgecolor='white', width=0.6)
ax1.bar_label(bars, padding=4, fontsize=10, fontweight='bold')
ax1.set_ylabel('Count')
ax1.set_title('Preference-SFT Dataset Composition', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(_values) * 1.3 if _values and max(_values) > 0 else 10)
ax1.grid(axis='y', alpha=0.3)

# ── Right: training loss curve (or fall back to train/valid bars) ─────
if train_losses or val_losses:
    if train_losses:
        ax2.plot(train_iters, train_losses, 'b-o', ms=3, lw=1.5, label='Train loss')
    if val_losses:
        ax2.plot(val_iters, val_losses, 'r-s', ms=7, lw=1.5, label='Val loss')
        ax2.axhline(min(val_losses), color='r', lw=0.8, ls='--', alpha=0.5,
                    label=f'Best val {min(val_losses):.4f}')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Loss')
    ax2.set_title('Preference-SFT Loss', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
else:
    _split_labels = ['Train', 'Valid']
    _split_values = [len(train_samples), len(valid_samples_dpo)]
    _split_colors = ['#3498db', '#f39c12']
    bars2 = ax2.bar(_split_labels, _split_values, color=_split_colors, edgecolor='white', width=0.4)
    ax2.bar_label(bars2, padding=4, fontsize=12, fontweight='bold')
    ax2.set_ylabel('Samples')
    ax2.set_title('Train / Valid Split', fontsize=13, fontweight='bold')
    ax2.set_ylim(0, max(_split_values) * 1.3 if _split_values and max(_split_values) > 0 else 10)
    ax2.grid(axis='y', alpha=0.3)

fig.suptitle(
    f'Preference-Filtered SFT — {len(all_dpo_pairs)} pairs  ·  '
    f'{DPO_ITERS} iters  ·  LR {LEARNING_RATE:.0e}',
    fontsize=13, fontweight='bold', y=1.02,
)

# ── Warning overlay ────────────────────────────────────────────────
# Triggers when early train-loss is unhealthy: NaN, an explosion, or
# a starting value way outside the sane 0.5–2.0 band. Renders a big
# coloured banner on the PNG so the failure is impossible to miss.
_warning_msg = None
_warning_color = '#c0392b'   # red
if _loss_nan:
    _at = f'iter {_first_nan_iter}' if _first_nan_iter is not None else 'early'
    _warning_msg = (
        f'TRAINING FAILED — NaN train loss at {_at}\n'
        f'Expected sane train loss ~0.5–1.0\n'
        f'Adapter is unusable. Lower LR (now {LEARNING_RATE:.0e}) '
        f'or tighten MAX_SEQ_LEN.'
    )
elif _loss_exploded:
    _at = f'iter {_first_explode_iter}' if _first_explode_iter is not None else 'early'
    _warning_msg = (
        f'TRAINING FAILED — loss explosion at {_at} '
        f'(>{LOSS_EXPLODE_THRESHOLD})\n'
        f'Reduce LEARNING_RATE (now {LEARNING_RATE:.0e}).'
    )
elif _first_train_loss is not None and _first_train_loss > 2.0:
    _warning_color = '#e67e22'   # orange — suspicious but not fatal
    _warning_msg = (
        f'WARNING — first train loss {_first_train_loss:.2f} '
        f'(expected ~0.5–1.0)\n'
        f'Possible LR / data mismatch. Inspect the loss curve.'
    )
elif _first_train_loss is not None and _first_train_loss < 0.05:
    _warning_color = '#e67e22'
    _warning_msg = (
        f'WARNING — first train loss {_first_train_loss:.4f} is suspiciously low\n'
        f'Likely the training data is too easy or already memorized.'
    )

if _warning_msg:
    fig.text(
        0.5, 0.5, _warning_msg,
        ha='center', va='center',
        fontsize=15, fontweight='bold', color='white',
        bbox=dict(facecolor=_warning_color, alpha=0.95,
                  boxstyle='round,pad=1.0', edgecolor='black', linewidth=1.5),
        zorder=100,
    )

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

## Summary

In [ ]:
print('=' * 60)
print('notebook 12 — PREFERENCE-FILTERED SFT SUMMARY')
print('=' * 60)

print(f'\nData:')
print(f'  Rejects loaded:         {len(all_rejects)}')
print(f'  Close-but-wrong:        {len(filtered_rejects)}')
print(f'  DPO pairs from rejects: {len(dpo_pairs)}')
print(f'  Fresh DPO pairs:        {len(fresh_pairs)}')
print(f'  Total DPO pairs:        {len(all_dpo_pairs)}')

print(f'\nTraining:')
print('  Objective: ' + ('preference loss (CE + margin hinge vs rejected)'
                         if USE_PREFERENCE_LOSS else 'chosen-only SFT'))
if _training_failed:
    if _loss_nan:
        print(f'  Status: FAILED — NaN loss')
    elif _loss_exploded:
        print(f'  Status: FAILED — loss explosion')
    else:
        print(f'  Status: FAILED')
else:
    print(f'  Status: OK ({len(train_iters)} iters logged)')
    if train_losses:
        print(f'  Final train_loss: {train_losses[-1]:.4f}')
    if val_losses:
        best_v = min(val_losses)
        best_vi = val_iters[val_losses.index(best_v)]
        print(f'  Best   val_loss:  {best_v:.4f}  (at iter {best_vi})')

print(f'\nModel:')
print(f'  SFT base:  {SFT_MODEL}')
if not _training_failed and DPO_FUSED.exists():
    size_gb = sum(f.stat().st_size for f in DPO_FUSED.rglob('*') if f.is_file()) / 1e9
    print(f'  Fused:     {DPO_FUSED} ({size_gb:.1f} GB)')
else:
    print(f'  Fused:     not created (training failed or fuse skipped)')

print(f'\nNext steps:')
if _training_failed:
    print(f'  1. Inspect the training output above — likely cause: truncation')
    print(f'     or LR too high. The pre-flight filter cell shows truncated counts.')
    print(f'  2. Lower LEARNING_RATE (currently {LEARNING_RATE:.0e}) or tighten')
    print(f'     MAX_SEQ_LEN (currently {MAX_SEQ_LEN}), then re-run.')
else:
    print(f'  1. Run NB12 (evaluation) to compare SFT vs preference-SFT model quality.')
    print(f'  2. Run NB24 (packaging) — it picks up the fused model automatically.')
print('=' * 60)